# Run 1 — YOLO11m Baseline Training

**Epic:** TTV-118 | **Config:** `yolo11m_baseline.yaml`

Este notebook entrena YOLO11m con configuración conservadora sobre dataset v1 (sin flip).

**Requisitos:** GPU runtime (T4 o superior)

---

## 0. Verificar GPU

In [ ]:
import torch

assert torch.cuda.is_available(), "ERROR: No GPU detectada. Ve a Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Instalar paquete desde GitHub

In [ ]:
!pip install -q git+https://github.com/SrPabvliss/th-cycling-photo-ai.git
!pip install -q roboflow ultralytics

## 2. Montar Google Drive (para guardar pesos)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUTPUT = '/content/drive/MyDrive/cycling-photo-ai/experiments'
os.makedirs(DRIVE_OUTPUT, exist_ok=True)
print(f"Output dir: {DRIVE_OUTPUT}")

## 3. Descargar dataset v1 desde Roboflow

In [ ]:
from roboflow import Roboflow

# --- CONFIGURAR API KEY ---
ROBOFLOW_API_KEY = "xOdnFACkI2vaUzBKVRic"  # TODO: mover a secrets de Colab

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace("titan-ca4ce").project("titan-detection-jedpa")
version = project.version(7)  # v7 = ai_phase_v1_no_flip

dataset = version.download("yolov11", location="/content/dataset_v1")
print(f"Dataset descargado en: /content/dataset_v1")

## 4. Verificar dataset

In [ ]:
from pathlib import Path

dataset_dir = Path("/content/dataset_v1")

for split in ["train", "valid", "test"]:
    imgs = list((dataset_dir / split / "images").glob("*.*"))
    lbls = list((dataset_dir / split / "labels").glob("*.txt"))
    print(f"{split}: {len(imgs)} images, {len(lbls)} labels")

# Mostrar data.yaml
print("\n--- data.yaml ---")
print((dataset_dir / "data.yaml").read_text())

## 5. Configurar reproducibilidad

In [ ]:
import os
import random
import numpy as np
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

print(f"Seed: {SEED} — reproducibility configured")

## 6. Entrenar YOLO11m — Run 1 Baseline

Config: `configs/training/yolo11m_baseline.yaml`

Hiperparámetros conservadores — sin mixup, sin copy-paste, cls_pw=1.0

In [ ]:
from ultralytics import YOLO

RUN_NAME = "run1_yolo11m_baseline"

model = YOLO("yolo11m.pt")

results = model.train(
    # Dataset
    data="/content/dataset_v1/data.yaml",

    # Training
    epochs=200,
    patience=30,
    imgsz=640,
    batch=16,
    optimizer="SGD",
    lr0=0.01,
    lrf=0.01,
    cos_lr=True,
    warmup_epochs=3,

    # Augmentation (conservative — runtime only)
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4,
    degrees=7.0,
    translate=0.1,
    scale=0.5,
    fliplr=0.0,   # v1 = sin flip
    flipud=0.0,   # NUNCA
    mosaic=1.0,
    close_mosaic=10,
    mixup=0.0,    # baseline = sin mixup
    cutmix=0.0,
    cls_pw=1.0,   # default class weight

    # Output
    save_json=True,
    deterministic=True,
    seed=SEED,
    project="/content/experiments",
    name=RUN_NAME,
)

## 7. Revisar resultados

In [ ]:
from pathlib import Path
import pandas as pd

run_dir = Path(f"/content/experiments/{RUN_NAME}")

# Métricas finales
print("=== Métricas finales ===")
for key, val in results.results_dict.items():
    print(f"  {key}: {val:.4f}")

# Training curve
results_csv = run_dir / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print(f"\nEpochs entrenados: {len(df)}")
    print(f"Mejor mAP@0.5: {df['metrics/mAP50(B)'].max():.4f} (epoch {df['metrics/mAP50(B)'].idxmax()})")
    print(f"Mejor mAP@0.5:0.95: {df['metrics/mAP50-95(B)'].max():.4f} (epoch {df['metrics/mAP50-95(B)'].idxmax()})")

In [ ]:
# Visualizar curvas de entrenamiento
import matplotlib.pyplot as plt

if results_csv.exists():
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    # Loss
    axes[0].plot(df['train/box_loss'], label='train box')
    axes[0].plot(df['train/cls_loss'], label='train cls')
    axes[0].plot(df['val/box_loss'], label='val box', linestyle='--')
    axes[0].plot(df['val/cls_loss'], label='val cls', linestyle='--')
    axes[0].set_title('Loss')
    axes[0].legend()
    axes[0].set_xlabel('Epoch')

    # mAP
    axes[1].plot(df['metrics/mAP50(B)'], label='mAP@0.5')
    axes[1].plot(df['metrics/mAP50-95(B)'], label='mAP@0.5:0.95')
    axes[1].axhline(y=0.80, color='r', linestyle=':', label='Target 0.80')
    axes[1].set_title('mAP')
    axes[1].legend()
    axes[1].set_xlabel('Epoch')

    # Precision/Recall
    axes[2].plot(df['metrics/precision(B)'], label='Precision')
    axes[2].plot(df['metrics/recall(B)'], label='Recall')
    axes[2].set_title('Precision / Recall')
    axes[2].legend()
    axes[2].set_xlabel('Epoch')

    plt.tight_layout()
    plt.savefig(run_dir / 'training_curves.png', dpi=150)
    plt.show()

## 8. Validación per-class

In [ ]:
# Evaluar en validation set con best weights
best_model = YOLO(str(run_dir / "weights" / "best.pt"))
val_results = best_model.val(
    data="/content/dataset_v1/data.yaml",
    imgsz=640,
    save_json=True,
)

# Per-class metrics
class_names = ['bicycle', 'bicycle_text', 'clothes_text', 'competidor_number', 'cyclist',
               'cyclist_clothes', 'cyclist_with_bike', 'helmet', 'helmet_text', 'objects']

print("\n=== Per-class AP@0.5 ===")
for i, name in enumerate(class_names):
    ap50 = val_results.box.ap50[i] if i < len(val_results.box.ap50) else 0
    ap = val_results.box.ap[i] if i < len(val_results.box.ap) else 0
    print(f"  {name:25s} AP@0.5={ap50:.4f}  AP@0.5:0.95={ap:.4f}")

## 9. Mostrar confusion matrix e imágenes de referencia

In [ ]:
from IPython.display import Image, display

# Confusion matrix
cm_path = run_dir / "confusion_matrix_normalized.png"
if cm_path.exists():
    print("Confusion Matrix (normalized):")
    display(Image(filename=str(cm_path), width=800))

# PR curves
pr_path = run_dir / "PR_curve.png"
if pr_path.exists():
    print("\nPR Curves:")
    display(Image(filename=str(pr_path), width=800))

# Sample predictions
pred_path = run_dir / "val_batch0_pred.jpg"
if pred_path.exists():
    print("\nSample predictions:")
    display(Image(filename=str(pred_path), width=800))

## 10. Guardar en Google Drive

In [ ]:
import shutil

drive_run_dir = Path(DRIVE_OUTPUT) / RUN_NAME

# Copiar todo el directorio del run a Drive
if drive_run_dir.exists():
    shutil.rmtree(drive_run_dir)

shutil.copytree(run_dir, drive_run_dir)

# Verificar
weights_size = (drive_run_dir / "weights" / "best.pt").stat().st_size / 1e6
print(f"Guardado en: {drive_run_dir}")
print(f"best.pt: {weights_size:.1f} MB")
print(f"\nArchivos guardados:")
for f in sorted(drive_run_dir.rglob("*")):
    if f.is_file():
        print(f"  {f.relative_to(drive_run_dir)} ({f.stat().st_size / 1e6:.1f} MB)")

## 11. Resumen para EXPERIMENT_LOG.md

Copia este output al experiment log del proyecto.

In [ ]:
print("="*60)
print("RESUMEN PARA EXPERIMENT_LOG.md")
print("="*60)
print(f"""\n### Run 1 — YOLO11m Baseline""")
print(f"- **Fecha:** {pd.Timestamp.now().strftime('%Y-%m-%d')}")
print(f"- **Config:** yolo11m_baseline.yaml")
print(f"- **Dataset:** v1 (sin flip), Roboflow v7")
print(f"- **GPU:** {torch.cuda.get_device_name(0)}")
print(f"- **Epochs entrenados:** {len(df)}")
print(f"- **Mejor epoch:** {df['metrics/mAP50(B)'].idxmax()}")
print(f"")
print(f"| Métrica | Valor |")
print(f"|---|---|")
print(f"| mAP@0.5 | {df['metrics/mAP50(B)'].max():.4f} |")
print(f"| mAP@0.5:0.95 | {df['metrics/mAP50-95(B)'].max():.4f} |")
print(f"| Precision | {df['metrics/precision(B)'].max():.4f} |")
print(f"| Recall | {df['metrics/recall(B)'].max():.4f} |")
print(f"")
print(f"**Per-class AP@0.5:**")
print(f"")
print(f"| Clase | AP@0.5 |")
print(f"|---|---|")
for i, name in enumerate(class_names):
    ap50 = val_results.box.ap50[i] if i < len(val_results.box.ap50) else 0
    print(f"| {name} | {ap50:.4f} |")
print(f"\nPesos guardados en: {drive_run_dir}/weights/best.pt")